In [2]:
import logging
import os

import xarray as xr
import numpy as np
import pcraster as pcr
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import resevoir_functions as rf

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

In [3]:
def estimate_discharge_for_environmental_flow(self, channelStorage):
    # z-score for 90th percentile; sets the environmental flow threshold
    z_score = 1.2816

    # Long-term variance of discharge using an online Welford algorithm
    varDischarge = self.m2tDischarge / \
                   pcr.max(1.,
                   pcr.min(self.maxTimestepsToAvgDischargeLong, self.timestepsToAvgDischarge) - 1.)
    stdDischarge = pcr.max(varDischarge**0.5, 0.0)

    # Floor at 10% of avg discharge to prevent flip-flop near the threshold
    minDischargeForEnvironmentalFlow = pcr.max(0.0, self.avgDischarge - z_score * stdDischarge)
    factor = 0.10
    minDischargeForEnvironmentalFlow = pcr.max(factor * self.avgDischarge, minDischargeForEnvironmentalFlow)
    minDischargeForEnvironmentalFlow = pcr.max(0.0, minDischargeForEnvironmentalFlow)

    return minDischargeForEnvironmentalFlow

#This function is most likely redundant as we have the net env flow from the netCDF files.

In [23]:
#loading data
def load_data(file_path):
    try:
        data = xr.open_dataset(file_path)
        logger.info(f"Data loaded successfully from {file_path}")
        return data
    except Exception as e:
        logger.error(f"Error loading data from {file_path}: {e}")
        return None

file_path = os.path.join(os.getcwd(), 'Data', 'POINTDATA') + os.sep

latitude = 37.625072479248            #hardcoded coordinates for a dam used in Jennies model as well. Later the model should be improved to automatically get the coordinates of the dam we want to model
longitude = -122.458351135254


# Monthly averages; daily resolution can be re-run for a select region
discharge = load_data(file_path + "sos_resout_final_monthAvg_output.nc").sel(lat=latitude, lon=longitude, method='nearest').to_dataframe().reset_index().drop(columns={'lon', 'lat'})
inflow = load_data(file_path + "soswaterInflow_annuaTot_output.nc").sel(lat=latitude, lon=longitude, method='nearest').to_dataframe().reset_index().drop(columns={'lon', 'lat'})  # yearly totals
demand_tot = load_data(file_path + 'sosreduction_demand_dailyTot_output.nc').sel(lat=latitude, lon=longitude, method='nearest').to_dataframe().reset_index().drop(columns={'lon', 'lat'})
env_flow = load_data(file_path + 'soswater_env_flow_monthTot_output.nc').sel(lat=latitude, lon=longitude, method='nearest').to_dataframe().reset_index().drop(columns={'lon', 'lat'})

Storage_validation = load_data(file_path + 'sosStor_check_dailyTot_output.nc').sel(lat=latitude, lon=longitude, method='nearest').to_dataframe().reset_index().drop(columns={'lon', 'lat'}) #also sets storage values at t=0


# Calculate the number of days in each month and the monthly inflow based on the daily inflow
inflow['year'] = inflow['time'].dt.year

output = pd.merge(demand_tot, discharge, on='time')
output = pd.merge(output, Storage_validation, on='time')
output = pd.merge(output, env_flow, on='time')

output['year'] = output['time'].dt.year
output = pd.merge(output, inflow[['year', 'soswater_inflow']], on='year')

output['days_in_month'] = output['time'].dt.days_in_month
output['inflow_monthly'] = (output['soswater_inflow'] / 365) * output['days_in_month']
output['modelled_storage'] = 0.0
output['model_release'] = 0.0
output['flood'] = 0.0
output['conservation'] = 0.0
output['model_current_storage'] = 0.0
output['reduction_factor_model'] = 0.0
output['day'] = range(0, len(output))




df = output.copy()
cap_215 = 23.5 * 1e6 #capacity of the dam in m3, used for normalization in the plots. We dont want this hardcoded in the final model.
avg_inflow = inflow['soswater_inflow'].mean()


2026-05-16 13:26:53,710 INFO Data loaded successfully from /home/pablosnackbar/resevoir_study/Data/POINTDATA/sos_resout_final_monthAvg_output.nc
2026-05-16 13:26:53,726 INFO Data loaded successfully from /home/pablosnackbar/resevoir_study/Data/POINTDATA/soswaterInflow_annuaTot_output.nc


2026-05-16 13:26:53,789 INFO Data loaded successfully from /home/pablosnackbar/resevoir_study/Data/POINTDATA/sosreduction_demand_dailyTot_output.nc
2026-05-16 13:26:53,866 INFO Data loaded successfully from /home/pablosnackbar/resevoir_study/Data/POINTDATA/soswater_env_flow_monthTot_output.nc
2026-05-16 13:26:53,943 INFO Data loaded successfully from /home/pablosnackbar/resevoir_study/Data/POINTDATA/sosStor_check_dailyTot_output.nc


In [24]:
df

,time,soswater_RF_demand,sos_reservoir_outflow_end,sos_storage_check,soswater_env_flow,year,soswater_inflow,days_in_month,inflow_monthly,modelled_storage,model_release,flood,conservation,model_current_storage,reduction_factor_model,day
0,1979-01-31,23500000.0,4021.334229,23500000.0,1.442840,1979,41645380.0,31,3.537005e+06,0.0,0.0,0.0,0.0,0.0,0.0,0
1,1979-02-28,23500000.0,4336.654297,23500000.0,1.405397,1979,41645380.0,28,3.194714e+06,0.0,0.0,0.0,0.0,0.0,0.0,1
2,1979-03-31,23500000.0,4816.763672,23500000.0,1.728237,1979,41645380.0,31,3.537005e+06,0.0,0.0,0.0,0.0,0.0,0.0,2
3,1979-04-30,23500000.0,5186.385742,23500000.0,1.800828,1979,41645380.0,30,3.422908e+06,0.0,0.0,0.0,0.0,0.0,0.0,3
4,1979-05-31,23500000.0,4849.555176,23500000.0,1.927865,1979,41645380.0,31,3.537005e+06,0.0,0.0,0.0,0.0,0.0,0.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
535,2023-08-31,17700000.0,0.000000,17700000.0,2.699958,2023,41039172.0,31,3.485519e+06,0.0,0.0,0.0,0.0,0.0,0.0,535
536,2023-09-30,17700000.0,0.000000,17700000.0,2.574645,2023,41039172.0,30,3.373083e+06,0.0,0.0,0.0,0.0,0.0,0.0,536
537,2023-10-31,17700000.0,0.000000,17700000.0,2.620943,2023,41039172.0,31,3.485519e+06,0.0,0.0,0.0,0.0,0.0,0.0,537
538,2023-11-30,17700000.0,238.140457,17700000.0,2.496387,2023,41039172.0,30,3.373083e+06,0.0,0.0,0.0,0.0,0.0,0.0,538


sos_reservoir_outflow_end
time       lat       lon                                   
1979-01-31 43.958332 -124.958336                        NaN
                     -124.875000                        NaN
                     -124.791664                        NaN
                     -124.708336                        NaN
                     -124.625000                        NaN